In [14]:
import os 
from dotenv import load_dotenv

load_dotenv()
if "GROQ_API_KEY" in os.environ:
    print("Groq API key detected and loaded")
else:
    print("Error: Groq API key not found in .env file.")

Groq API key detected and loaded


**The Schema**

We use a Pydantic model to make sure the LLM gives us a clean object.

In [15]:
from pydantic import BaseModel, Field
from typing import Optional, List

class DocClassification(BaseModel):
    category: str= Field(description="One of the 20 categories")
    confidence: float =Field(description="The 0.0 - 1.0 condidence score")
    reason: str = Field(description="Brief explanation for the choice.")

**The Agent State**

This tracks the document as it moves throught the nodes. We use a TypedDict

In [16]:
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    file_path: str
    content: str
    keyword_source: List[str]
    final_output: Optional[DocClassification]
    model_tier: str
    is_valid: bool



In [17]:
Categories = [
    "Legal_Contract",       # Agreements, NDAs, and Leases
    "Legal_Litigation",     # Court filings and summons
    "Finance_Invoice",      # Bills and payment requests
    "Finance_Tax",          # Tax returns and filings
    "Finance_Statement",    # P&L and Balance Sheets
    "HR_Resume",            # CVs and applications
    "HR_Policy",            # Handbooks and conduct guides
    "Tech_Manual",          # User guides and instructions
    "Tech_Architecture",    # System designs and diagrams
    "Tech_API_Doc",         # Developer documentation
    "Ops_Logistics",        # Shipping and manifests
    "Ops_SOP",             # Standard Operating Procedures
    "Mkt_Strategy",         # Brand and campaign plans
    "Med_Clinical",         # Patient notes and history
    "PM_Project_Plan"       # Gantt charts and milestones
    "Self-help"
]

**The Preprocessor Node**

This is a **Filter Node** that elminiates items that do not meet criteria. The criteria for this Agent is that the chosen document should be a non-empty English pdf or word document.

We will use the **docling**(by IBM) library to convert the documents into markdown which is easier for LLMs to understand.

This node thus:
- Takes the **AgentState**
- Checks if the **content* is empty
- uses the **langdetect** library to check if the content is in English.
- Sets **is_invalid* to TRUE or FALSE


In [18]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

def preprocessor_node(state: AgentState):
    print("---NODE: PREPROCESSOR (Optimized)---")
    file_path = state.get("file_path")
    
    try:
        # 1. Setup Lean Pipeline Options
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = False  # Disable OCR to save RAM
        pipeline_options.do_table_structure = True # Keep tables if relevant
        
        # 2. Initialize Converter with Options
        converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )
        
        # 3. Convert (Docling handles the file stream)
        result = converter.convert(file_path)
        
        # 4. Surgical Text Capture
        # We only take the first 10,000 characters. 
        # This is usually the first 5-8 pages—plenty for classification.
        text_content = result.document.export_to_markdown()
        truncated_content = text_content[:10000]
        
        if not truncated_content.strip():
            return {"is_valid": False, "content": "Error: Extraction resulted in empty text."}
            
        return {"content": truncated_content, "is_valid": True}
        
    except Exception as e:
        print(f"Extraction failed: {e}")
        return {"is_valid": False, "content": f"Hardware/Logic Error: {str(e)}"}

**The Keyword Scorer Node**

This node contains a simple dictionary of anchor words for a few categories
It checks the content of these words and saves the matches into the **keyword_source**.


In [19]:
def keyword_node(state: AgentState):
    print("NODE: KEYWORD SCORER")
    content = state["content"].lower()

    category_hints = {
        # 1-10: LEGAL & COMPLIANCE
        "Legal_Contract": ["agreement", "termination", "indemnity", "jurisdiction"],
        "Legal_NDA": ["non-disclosure", "confidential information", "trade secret"],
        "Compliance_GDPR": ["data subject", "processing", "privacy policy", "consent"],
        
        # 11-20: FINANCE & ACCOUNTING
        "Finance_Invoice": ["bill to", "invoice number", "amount due", "remit"],
        "Finance_Audit": ["reconciliation", "internal control", "ledger", "variance"],
        "Finance_Tax": ["tax return", "deduction", "irs", "withholding", "vat"],
        
        # 21-30: HUMAN RESOURCES
        "HR_Resume": ["education", "experience", "skills", "employment history"],
        "HR_Policy": ["employee handbook", "code of conduct", "leave policy"],
        "HR_Performance": ["appraisal", "KPI", "feedback", "review cycle"],
        
        # 31-40: TECHNICAL & IT
        "IT_Architecture": ["cloud", "microservices", "infrastructure", "latency"],
        "IT_Security": ["vulnerability", "firewall", "encryption", "penetration"],
        "IT_API_Doc": ["endpoint", "payload", "request body", "authentication"],
        
        # 41-50: OPERATIONS & MARKETING
        "Ops_Logistics": ["bill of lading", "shipping", "warehouse", "inventory"],
        "Mkt_Strategy": ["target audience", "brand identity", "campaign", "ROI"],
        "Mkt_Research": ["demographics", "focus group", "market share", "survey"]
    }

    found_hints = []
    for category, keywords in category_hints.items():
        for word in keywords:
            if word in content:
                found_hints.append(f"Potential category: {category} (Matched: {word})")
                break

    return {"keyword_source": found_hints}



**Base Classifier Node (Groq 8b)**

This is the first LLM node that is fast, uses the keyword_source to produce a specific category and confidence score. 

We use **.with_structured_output* to force the LLM to follow the Pydantic schema.

In [ ]:
from langchain_groq import ChatGroq

#initializes the "Base" model
base_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)

#bind the pydantic class to the LLM
structured_llm = base_llm.with_structured_output(DocClassification)

In [21]:
#Build the classifier node
def base_classifier(state: AgentState):
    print("NODE: BASE CLASSIFIER(8B)")

    categories_str= ",".join(Categories)

    role = "You are an experienced Document specialist with 12 years of experience" \
    "in classifying documents from a variety of fields and domains."

    prompt = f"""
    {role}

    Your task is to classify the text into ONE of these categories: {categories_str}

    CONTEXT:
    Rule-based hints detected: {state['keyword_source']}
    Use these hints as a guide, but prioritize the actual text content.

    CONSTRAINTS:
    You must return a confidence score between 0.0 to 1.0.
    If the document is ambigious, provide your best guess but reflect it in a lower confidence score.
    The 'reason' field must explain linguistic evidence found in the text.

    REASONING INSTRUCTIONS:
    Scan the headers and introductory paragraphs for intent.
    Cross reference identified keywords with the allowed category list.
    Determine the primary classification and assign a confidence weight.
    
    """

    #Execution
    response = structured_llm.invoke([
        ("system", prompt),
        ("human", f"Document Content:\n{state['content'][:6000]}")
    ])

    return {"final_output":response, "model_tier":"base"}


**Compile the Graph**

We will use **StateGraph** to define the flow.

In [22]:
from langgraph.graph import StateGraph, START, END

#Initialize the graph with our AgentState
graph= StateGraph(AgentState)

graph.add_node("Preprocessor", preprocessor_node)
graph.add_node("Keywords", keyword_node)
graph.add_node("Base-classifier", base_classifier)

#Define the flow
graph.add_edge(START, "Preprocessor")
graph.add_edge("Preprocessor","Keywords")
graph.add_edge("Keywords","Base-classifier")
graph.add_edge("Base-classifier", END)

app = graph.compile()
print("Graph Compiled Successfully.")

Graph Compiled Successfully.


In [23]:
inputs = {
    "file_path": "The+48+Laws+Of+Power.pdf",
    "categories": Categories,
    "content": "",
    "keyword_source": [],
    "is_valid": True,
    "model_tier": "initial"
}

#Execute the graph
final_state = app.invoke(inputs)

#The results
print(f"--- FINAL RESULT ---")
print(f"Category: {final_state['final_output'].category}")
print(f"Confidence: {final_state['final_output'].confidence}")
print(f"Model Used: {final_state['model_tier']}")
print(f"Reasoning: {final_state['final_output'].reason}")


---NODE: PREPROCESSOR (Optimized)---


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1359.40it/s]
Stage preprocess failed for run 1, pages [3]: std::bad_alloc
Stage preprocess failed for run 1, pages [5]: std::bad_alloc
Stage preprocess failed for run 1, pages [6]: std::bad_alloc
Stage preprocess failed for run 1, pages [8]: std::bad_alloc
Stage preprocess failed for run 1, pages [9]: std::bad_alloc
Stage preprocess failed for run 1, pages [10]: std::bad_alloc
Stage preprocess failed for run 1, pages [11]: std::bad_alloc
Stage preprocess failed for run 1, pages [12]: std::bad_alloc
Stage preprocess failed for run 1, pages [13]: std::bad_alloc
Stage preprocess failed for run 1, pages [14]: std::bad_alloc
Stage preprocess failed for run 1, pages [15]: std::bad_alloc
Stage preprocess failed for run 1, pages [16]: std::bad_alloc
Stage preprocess failed for run 1, pages [17]: std::bad_alloc
Stage preprocess failed for run 1, pages [18]: std::bad_alloc
Stage preprocess failed for run 1, pages [19]: std::bad_alloc
Stage 

NODE: KEYWORD SCORER
NODE: BASE CLASSIFIER(8B)


BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}